# Reto 8: Analizador de Precios de Acciones

## Programacion para Ciencia de Datos
### Instituto Politecnico Nacional
### Febrero - Julio 2026

---

## Contexto del Problema

Una casa de bolsa necesita un sistema para analizar el comportamiento historico de precios de acciones usando Pandas Series. El sistema debe calcular metricas de rendimiento, identificar tendencias y generar alertas.

La tarea consiste en crear un analizador de acciones que procese datos de precios y genere insights utiles para inversionistas.

## Importaciones

In [1]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional

---
## PARTE 1: Analisis Estadistico Basico (30%)

Esta parte calcula las metricas descriptivas fundamentales de los precios y los rendimientos diarios.

In [2]:
def estadisticas_basicas(precios: pd.Series) -> Dict:
    """
    Calcula estadisticas descriptivas de los precios.

    Usamos los metodos nativos de Series para obtener cada metrica.
    .iloc[-1] nos da el ultimo precio (el precio actual de cierre),
    y el rango es simplemente la distancia entre el maximo y el minimo.

    Retorna:
        {
            "precio_actual": float,
            "precio_minimo": float,
            "precio_maximo": float,
            "precio_promedio": float,
            "precio_mediana": float,
            "desviacion_std": float,
            "rango": float,
            "dias_analizados": int
        }
    """
    resultado = {
        # El precio actual es el ultimo dato disponible en la serie
        "precio_actual": float(precios.iloc[-1]),
        "precio_minimo": float(precios.min()),
        "precio_maximo": float(precios.max()),
        "precio_promedio": float(precios.mean()),
        "precio_mediana": float(precios.median()),
        "desviacion_std": float(precios.std()),
        # El rango nos dice cuanto se movio el precio en todo el periodo
        "rango": float(precios.max() - precios.min()),
        "dias_analizados": int(len(precios))
    }
    return resultado

In [3]:
def calcular_rendimientos(precios: pd.Series) -> pd.Series:
    """
    Calcula el rendimiento diario porcentual.

    Usamos .pct_change() que aplica la formula (valor_hoy / valor_ayer) - 1
    sobre toda la serie de forma vectorizada. Multiplicamos por 100 para
    expresarlo en porcentaje. El primer valor sera NaN porque no tiene
    dia anterior con que compararse.

    Formula: ((precio_hoy / precio_ayer) - 1) * 100

    Retorna: Series con rendimientos diarios (%)
    """
    return precios.pct_change() * 100

In [4]:
def analisis_rendimientos(rendimientos: pd.Series) -> Dict:
    """
    Analiza los rendimientos calculados.

    Eliminamos el NaN inicial antes de operar. El rendimiento total
    se obtiene sumando todos los rendimientos diarios. Usamos
    .idxmax() e .idxmin() para obtener la fecha exacta del mejor
    y peor dia respectivamente.

    Retorna:
        {
            "rendimiento_total": float,
            "rendimiento_promedio": float,
            "mejor_dia": tuple (fecha, rendimiento),
            "peor_dia": tuple (fecha, rendimiento),
            "dias_positivos": int,
            "dias_negativos": int,
            "volatilidad": float
        }
    """
    # Eliminamos el NaN del primer dia para no contaminar los calculos
    rend_limpio = rendimientos.dropna()

    fecha_mejor = rend_limpio.idxmax()
    fecha_peor = rend_limpio.idxmin()

    resultado = {
        "rendimiento_total": float(rend_limpio.sum()),
        "rendimiento_promedio": float(rend_limpio.mean()),
        # Guardamos fecha formateada y el valor de rendimiento de ese dia
        "mejor_dia": (str(fecha_mejor.date()), float(rend_limpio[fecha_mejor])),
        "peor_dia": (str(fecha_peor.date()), float(rend_limpio[fecha_peor])),
        # Contamos dias con rendimiento positivo y negativo
        "dias_positivos": int((rend_limpio > 0).sum()),
        "dias_negativos": int((rend_limpio < 0).sum()),
        # La volatilidad es la desviacion estandar de los rendimientos
        "volatilidad": float(rend_limpio.std())
    }
    return resultado

---
## PARTE 2: Indicadores Tecnicos (35%)

Los indicadores tecnicos permiten identificar tendencias y niveles de soporte/resistencia en el precio.

In [5]:
def media_movil(precios: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la media movil simple (SMA).

    .rolling(window) crea una ventana deslizante de N periodos.
    Al llamar .mean() sobre ella, obtenemos el promedio de los
    ultimos N precios para cada fecha. Los primeros N-1 valores
    seran NaN porque no hay suficientes datos para calcular el promedio.

    Args:
        precios: Serie de precios
        ventana: Numero de periodos para el promedio

    Retorna: Serie con la media movil
    """
    return precios.rolling(window=ventana).mean()

In [6]:
def bandas_bollinger(precios: pd.Series, ventana: int = 20, num_std: int = 2) -> Dict:
    """
    Calcula las Bandas de Bollinger.

    La banda media es simplemente la SMA. Las bandas superior e inferior
    se construyen sumando o restando N desviaciones estandar a la banda
    media. Cuando el precio toca la banda superior, suele interpretarse
    como sobrecompra; cuando toca la inferior, como sobreventa.

    - Banda media: SMA de 'ventana' periodos
    - Banda superior: SMA + (num_std x desviacion estandar)
    - Banda inferior: SMA - (num_std x desviacion estandar)

    Retorna:
        {
            "banda_superior": pd.Series,
            "banda_media": pd.Series,
            "banda_inferior": pd.Series
        }
    """
    # Calculamos la media movil y la desviacion estandar movil
    sma = precios.rolling(window=ventana).mean()
    std_movil = precios.rolling(window=ventana).std()

    resultado = {
        "banda_superior": sma + (num_std * std_movil),
        "banda_media": sma,
        "banda_inferior": sma - (num_std * std_movil)
    }
    return resultado

In [7]:
def detectar_maximos_minimos(precios: pd.Series, ventana: int = 5) -> Dict:
    """
    Detecta maximos y minimos locales.

    Usamos una ventana centrada (center=True) de tamano 2*ventana+1.
    Un punto es maximo local si es igual al maximo dentro de esa ventana,
    y minimo local si es igual al minimo. El parametro center=True hace
    que la ventana se extienda tanto hacia atras como hacia adelante.

    Un maximo local: precio mayor que los 'ventana' dias anteriores y posteriores
    Un minimo local: precio menor que los 'ventana' dias anteriores y posteriores

    Retorna:
        {
            "maximos": pd.Series,
            "minimos": pd.Series
        }
    """
    # La ventana total es 2*ventana+1 para incluir el punto central
    tamano_ventana = 2 * ventana + 1

    rolling_max = precios.rolling(window=tamano_ventana, center=True).max()
    rolling_min = precios.rolling(window=tamano_ventana, center=True).min()

    # Un punto es maximo local cuando coincide con el maximo de su vecindad
    maximos = precios[precios == rolling_max]
    minimos = precios[precios == rolling_min]

    resultado = {
        "maximos": maximos,
        "minimos": minimos
    }
    return resultado

In [8]:
def clasificar_tendencia(precios: pd.Series, ventana: int = 10) -> str:
    """
    Clasifica la tendencia actual.

    Comparamos el precio actual contra la media movil. Si el precio
    esta por encima de la MA y ademas la MA viene subiendo (valor
    actual mayor que el anterior), la tendencia es alcista. Si esta
    por debajo y la MA baja, es bajista. En cualquier otro caso, el
    mercado esta en lateral sin tendencia definida.

    Criterios:
    - ALCISTA: Precio actual > MA y MA subiendo
    - BAJISTA: Precio actual < MA y MA bajando
    - LATERAL: Sin tendencia clara

    Retorna: "ALCISTA", "BAJISTA" o "LATERAL"
    """
    ma = media_movil(precios, ventana)

    precio_actual = precios.iloc[-1]
    ma_actual = ma.iloc[-1]
    ma_anterior = ma.iloc[-2]  # Para saber si la MA esta subiendo o bajando

    ma_subiendo = ma_actual > ma_anterior
    ma_bajando = ma_actual < ma_anterior

    if precio_actual > ma_actual and ma_subiendo:
        return "ALCISTA"
    elif precio_actual < ma_actual and ma_bajando:
        return "BAJISTA"
    else:
        return "LATERAL"

---
## PARTE 3: Sistema de Alertas (35%)

El sistema de alertas permite identificar oportunidades de compra/venta y movimientos anormales en el precio.

In [9]:
def generar_senales_trading(precios: pd.Series, ma_corta: int = 5, ma_larga: int = 20) -> pd.Series:
    """
    Genera senales de compra/venta basadas en cruces de medias moviles.

    La logica del cruce es: detectamos si la MA corta supera a la MA larga
    (cruce dorado = COMPRA) o si cae por debajo (cruce de la muerte = VENTA).
    Para detectar el cruce usamos una mascara booleana y la comparamos con
    su version desplazada un periodo (.shift(1)) para ver el cambio.

    - COMPRA: MA corta cruza MA larga hacia arriba
    - VENTA: MA corta cruza MA larga hacia abajo
    - MANTENER: Sin cruce

    Retorna: Serie con senales ("COMPRA", "VENTA", "MANTENER")
    """
    sma_corta = media_movil(precios, ma_corta)
    sma_larga = media_movil(precios, ma_larga)

    # Mascara: True donde la MA corta esta por encima de la MA larga
    corta_arriba = sma_corta > sma_larga

    # Cruce hacia arriba: hoy esta arriba pero ayer no lo estaba
    cruce_compra = corta_arriba & ~corta_arriba.shift(1).fillna(False)
    # Cruce hacia abajo: hoy esta abajo pero ayer estaba arriba
    cruce_venta = ~corta_arriba & corta_arriba.shift(1).fillna(False)

    # Inicializamos toda la serie como MANTENER
    senales = pd.Series("MANTENER", index=precios.index)
    senales[cruce_compra] = "COMPRA"
    senales[cruce_venta] = "VENTA"

    return senales

In [10]:
def alertas_precio(precios: pd.Series, umbral_cambio: float = 5.0) -> List[Dict]:
    """
    Genera alertas cuando hay cambios significativos.

    Calculamos los rendimientos diarios y filtramos aquellos donde el
    valor absoluto supera el umbral indicado. Cada alerta incluye la
    fecha, si fue subida o caida, y el porcentaje exacto de cambio.

    Args:
        precios: Serie de precios
        umbral_cambio: Porcentaje minimo para generar alerta

    Retorna: Lista de alertas con formato:
        [{"fecha": str, "tipo": "SUBIDA"/"CAIDA", "cambio": float}, ...]
    """
    alertas = []

    rendimientos = calcular_rendimientos(precios).dropna()

    # Filtramos los dias donde el movimiento fue mayor al umbral
    movimientos_grandes = rendimientos[rendimientos.abs() >= umbral_cambio]

    for fecha, cambio in movimientos_grandes.items():
        tipo = "SUBIDA" if cambio > 0 else "CAIDA"
        alertas.append({
            "fecha": str(fecha.date()),
            "tipo": tipo,
            "cambio": round(float(cambio), 2)
        })

    return alertas

In [11]:
def clasificar_volatilidad(rendimientos: pd.Series) -> str:
    """
    Clasifica el nivel de volatilidad del activo.

    Calculamos la desviacion estandar de los rendimientos, que mide
    que tan dispersos estan los retornos diarios. Una std alta indica
    que el precio se mueve mucho cada dia (mayor riesgo).

    Criterios (desviacion estandar de rendimientos):
    - BAJA:     < 1%
    - MEDIA:    1% - 3%
    - ALTA:     3% - 5%
    - MUY ALTA: > 5%

    Retorna: Clasificacion de volatilidad
    """
    std = rendimientos.dropna().std()

    if std < 1.0:
        return "BAJA"
    elif std < 3.0:
        return "MEDIA"
    elif std < 5.0:
        return "ALTA"
    else:
        return "MUY ALTA"

In [12]:
def generar_reporte_completo(precios: pd.Series, nombre_accion: str) -> Dict:
    """
    Genera un reporte completo de analisis integrando todas las funciones.

    Orquesta el llamado a las funciones de las tres partes anteriores
    para producir un resumen ejecutivo de la accion. La senal actual
    se toma del ultimo valor de la serie de senales de trading.

    Retorna:
        {
            "nombre": str,
            "periodo": {"inicio": str, "fin": str, "dias": int},
            "estadisticas": dict,
            "rendimientos": dict,
            "tendencia": str,
            "volatilidad": str,
            "senal_actual": str,
            "alertas_recientes": list
        }
    """
    # --- Parte 1 ---
    stats = estadisticas_basicas(precios)
    rendimientos = calcular_rendimientos(precios)
    analisis_rend = analisis_rendimientos(rendimientos)

    # --- Parte 2 ---
    tendencia = clasificar_tendencia(precios)

    # --- Parte 3 ---
    volatilidad = clasificar_volatilidad(rendimientos)
    senales = generar_senales_trading(precios)
    alertas = alertas_precio(precios)

    # La senal actual es la del ultimo dia disponible
    senal_actual = senales.iloc[-1]

    reporte = {
        "nombre": nombre_accion,
        "periodo": {
            "inicio": str(precios.index[0].date()),
            "fin": str(precios.index[-1].date()),
            "dias": len(precios)
        },
        "estadisticas": stats,
        "rendimientos": analisis_rend,
        "tendencia": tendencia,
        "volatilidad": volatilidad,
        "senal_actual": senal_actual,
        "alertas_recientes": alertas
    }
    return reporte

---
## Datos de Prueba

In [13]:
# Simular 60 dias de precios de una accion
np.random.seed(42)  # Para reproducibilidad

# Generar fechas en dias habiles (lunes a viernes)
fechas = pd.date_range(start='2024-01-01', periods=60, freq='B')

# Generar precios con tendencia y volatilidad realista
precio_inicial = 100
rendimientos_simulados = np.random.normal(0.002, 0.02, 60)
precios_simulados = precio_inicial * np.cumprod(1 + rendimientos_simulados)

PRECIOS_ACCION = pd.Series(
    precios_simulados.round(2),
    index=fechas,
    name='ACME Corp'
)

print("Precios de ACME Corp (primeros 10 dias):")
print(PRECIOS_ACCION.head(10))
print(f"\nTotal de dias: {len(PRECIOS_ACCION)}")

Precios de ACME Corp (primeros 10 dias):
2024-01-01    101.19
2024-01-02    101.12
2024-01-03    102.63
2024-01-04    105.96
2024-01-05    105.68
2024-01-08    105.39
2024-01-09    108.93
2024-01-10    110.82
2024-01-11    110.00
2024-01-12    111.42
Freq: B, Name: ACME Corp, dtype: float64

Total de dias: 60


In [14]:
# Datos adicionales para pruebas mas completas
np.random.seed(123)

# Accion con alta volatilidad
rend_volatil = np.random.normal(0, 0.05, 60)
precios_volatil = 50 * np.cumprod(1 + rend_volatil)
ACCION_VOLATIL = pd.Series(
    precios_volatil.round(2),
    index=fechas,
    name='VolatilTech'
)

# Accion con tendencia bajista
rend_bajista = np.random.normal(-0.005, 0.015, 60)
precios_bajista = 200 * np.cumprod(1 + rend_bajista)
ACCION_BAJISTA = pd.Series(
    precios_bajista.round(2),
    index=fechas,
    name='DeclineCorp'
)

print("Acciones disponibles para analisis:")
print(f"1. ACME Corp    - Precio actual: ${PRECIOS_ACCION.iloc[-1]:.2f}")
print(f"2. VolatilTech  - Precio actual: ${ACCION_VOLATIL.iloc[-1]:.2f}")
print(f"3. DeclineCorp  - Precio actual: ${ACCION_BAJISTA.iloc[-1]:.2f}")

Acciones disponibles para analisis:
1. ACME Corp    - Precio actual: $92.74
2. VolatilTech  - Precio actual: $59.42
3. DeclineCorp  - Precio actual: $138.97


---
## Funciones de Visualizacion

In [15]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de forma legible en consola."""
    print("=" * 70)
    print(f"           REPORTE DE ANALISIS: {reporte['nombre']}")
    print("=" * 70)

    periodo = reporte.get('periodo', {})
    print(f"\nPERIODO DE ANALISIS")
    print("-" * 40)
    print(f"Inicio: {periodo.get('inicio', 'N/A')}")
    print(f"Fin: {periodo.get('fin', 'N/A')}")
    print(f"Dias analizados: {periodo.get('dias', 'N/A')}")

    stats = reporte.get('estadisticas', {})
    print(f"\nESTADISTICAS DE PRECIO")
    print("-" * 40)
    print(f"Precio actual:   ${stats.get('precio_actual', 0):,.2f}")
    print(f"Precio minimo:   ${stats.get('precio_minimo', 0):,.2f}")
    print(f"Precio maximo:   ${stats.get('precio_maximo', 0):,.2f}")
    print(f"Precio promedio: ${stats.get('precio_promedio', 0):,.2f}")
    print(f"Precio mediana:  ${stats.get('precio_mediana', 0):,.2f}")
    print(f"Desv. estandar:  ${stats.get('desviacion_std', 0):,.2f}")
    print(f"Rango:           ${stats.get('rango', 0):,.2f}")

    rend = reporte.get('rendimientos', {})
    print(f"\nRENDIMIENTO")
    print("-" * 40)
    print(f"Rendimiento total: {rend.get('rendimiento_total', 0):+.2f}%")
    print(f"Rendimiento promedio diario: {rend.get('rendimiento_promedio', 0):+.3f}%")
    if rend.get('mejor_dia'):
        print(f"Mejor dia: {rend['mejor_dia'][0]} ({rend['mejor_dia'][1]:+.2f}%)")
    if rend.get('peor_dia'):
        print(f"Peor dia: {rend['peor_dia'][0]} ({rend['peor_dia'][1]:+.2f}%)")
    print(f"Dias positivos: {rend.get('dias_positivos', 0)}")
    print(f"Dias negativos: {rend.get('dias_negativos', 0)}")

    print(f"\nINDICADORES")
    print("-" * 40)
    print(f"Tendencia:    {reporte.get('tendencia', 'N/A')}")
    print(f"Volatilidad:  {reporte.get('volatilidad', 'N/A')}")
    print(f"Senal actual: {reporte.get('senal_actual', 'N/A')}")

    alertas = reporte.get('alertas_recientes', [])
    if alertas:
        print(f"\nALERTAS RECIENTES")
        print("-" * 40)
        for alerta in alertas[-5:]:
            signo = "+" if alerta['tipo'] == 'SUBIDA' else ""
            print(f"[{alerta['tipo']}] {alerta['fecha']}: {signo}{alerta['cambio']:.2f}%")

    print("\n" + "=" * 70)

In [16]:
def visualizar_precios_texto(precios: pd.Series, ancho: int = 50) -> None:
    """Visualizacion simple de precios en texto (ASCII chart)."""
    min_precio = precios.min()
    max_precio = precios.max()
    rango = max_precio - min_precio

    print(f"\nGrafico de precios: {precios.name}")
    print(f"Max: ${max_precio:.2f}")
    print("-" * (ancho + 10))

    for fecha, precio in precios.iloc[::3].items():
        posicion = int((precio - min_precio) / rango * ancho) if rango > 0 else ancho // 2
        barra = " " * posicion + "#"
        fecha_str = fecha.strftime('%m/%d') if hasattr(fecha, 'strftime') else str(fecha)[:5]
        print(f"{fecha_str} |{barra}")

    print("-" * (ancho + 10))
    print(f"Min: ${min_precio:.2f}")

---
## Prueba de Funciones Individuales

In [17]:
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

print("\n-- Estadisticas Basicas --")
stats = estadisticas_basicas(PRECIOS_ACCION)
for clave, valor in stats.items():
    print(f"  {clave}: {valor}")

print("\n-- Rendimientos (primeros 5) --")
rendimientos = calcular_rendimientos(PRECIOS_ACCION)
print(rendimientos.head())

print("\n-- Analisis de Rendimientos --")
analisis = analisis_rendimientos(rendimientos)
for clave, valor in analisis.items():
    print(f"  {clave}: {valor}")

PRUEBA DE FUNCIONES INDIVIDUALES

-- Estadisticas Basicas --
  precio_actual: 92.74
  precio_minimo: 86.67
  precio_maximo: 111.42
  precio_promedio: 96.88983333333334
  precio_mediana: 95.645
  desviacion_std: 7.128231646939469
  rango: 24.75
  dias_analizados: 60

-- Rendimientos (primeros 5) --
2024-01-01         NaN
2024-01-02   -0.069177
2024-01-03    1.493275
2024-01-04    3.244665
2024-01-05   -0.264251
Freq: B, Name: ACME Corp, dtype: float64

-- Analisis de Rendimientos --
  rendimiento_total: -7.748229086684423
  rendimiento_promedio: -0.1313259167234648
  mejor_dia: ('2024-02-13', 3.905831995719633)
  peor_dia: ('2024-02-21', -3.714554776603529)
  dias_positivos: 26
  dias_negativos: 33
  volatilidad: 1.823543256771936


In [18]:
print("\n-- Media Movil (5 dias, ultimos 5 valores) --")
ma5 = media_movil(PRECIOS_ACCION, 5)
print(ma5.tail())

print("\n-- Bandas de Bollinger (ultimos valores) --")
bandas = bandas_bollinger(PRECIOS_ACCION, 20, 2)
for nombre, serie in bandas.items():
    if serie is not None:
        print(f"  {nombre}: ${serie.iloc[-1]:.2f}")

print("\n-- Maximos y Minimos Locales --")
extremos = detectar_maximos_minimos(PRECIOS_ACCION)
print(f"  Maximos detectados: {len(extremos['maximos'])}")
print(extremos['maximos'])
print(f"  Minimos detectados: {len(extremos['minimos'])}")
print(extremos['minimos'])

print("\n-- Tendencia --")
tendencia = clasificar_tendencia(PRECIOS_ACCION)
print(f"  Tendencia actual: {tendencia}")

print("\n-- Senales de Trading --")
senales = generar_senales_trading(PRECIOS_ACCION)
print(senales.value_counts())

print("\n-- Alertas de Precio (umbral 3%) --")
alertas = alertas_precio(PRECIOS_ACCION, umbral_cambio=3.0)
for a in alertas:
    print(f"  [{a['tipo']}] {a['fecha']}: {a['cambio']:+.2f}%")

print("\n-- Clasificacion de Volatilidad --")
vol = clasificar_volatilidad(rendimientos)
print(f"  Volatilidad: {vol}")


-- Media Movil (5 dias, ultimos 5 valores) --
2024-03-18    88.776
2024-03-19    89.318
2024-03-20    89.986
2024-03-21    90.564
2024-03-22    91.134
Freq: B, Name: ACME Corp, dtype: float64

-- Bandas de Bollinger (ultimos valores) --
  banda_superior: $93.65
  banda_media: $89.85
  banda_inferior: $86.06

-- Maximos y Minimos Locales --
  Maximos detectados: 2
2024-01-12    111.42
2024-02-14     97.27
Name: ACME Corp, dtype: float64
  Minimos detectados: 3
2024-02-12    93.45
2024-02-22    89.76
2024-03-13    86.67
Name: ACME Corp, dtype: float64

-- Tendencia --
  Tendencia actual: ALCISTA

-- Senales de Trading --
MANTENER    57
COMPRA       3
Name: count, dtype: int64

-- Alertas de Precio (umbral 3%) --
  [SUBIDA] 2024-01-04: +3.24%
  [SUBIDA] 2024-01-09: +3.36%
  [CAIDA] 2024-01-18: -3.63%
  [CAIDA] 2024-01-19: -3.25%
  [SUBIDA] 2024-01-29: +3.13%
  [SUBIDA] 2024-02-13: +3.91%
  [CAIDA] 2024-02-21: -3.71%
  [CAIDA] 2024-03-08: -3.33%

-- Clasificacion de Volatilidad --
  Volat

---
## Reporte Completo

In [19]:
print("\nGENERANDO REPORTE COMPLETO PARA ACME CORP...\n")
reporte = generar_reporte_completo(PRECIOS_ACCION, "ACME Corp")
mostrar_reporte(reporte)


GENERANDO REPORTE COMPLETO PARA ACME CORP...

           REPORTE DE ANALISIS: ACME Corp

PERIODO DE ANALISIS
----------------------------------------
Inicio: 2024-01-01
Fin: 2024-03-22
Dias analizados: 60

ESTADISTICAS DE PRECIO
----------------------------------------
Precio actual:   $92.74
Precio minimo:   $86.67
Precio maximo:   $111.42
Precio promedio: $96.89
Precio mediana:  $95.64
Desv. estandar:  $7.13
Rango:           $24.75

RENDIMIENTO
----------------------------------------
Rendimiento total: -7.75%
Rendimiento promedio diario: -0.131%
Mejor dia: 2024-02-13 (+3.91%)
Peor dia: 2024-02-21 (-3.71%)
Dias positivos: 26
Dias negativos: 33

INDICADORES
----------------------------------------
Tendencia:    ALCISTA
Volatilidad:  MEDIA
Senal actual: COMPRA



In [20]:
# Grafico ASCII de precios
visualizar_precios_texto(PRECIOS_ACCION)


Grafico de precios: ACME Corp
Max: $111.42
------------------------------------------------------------
01/01 |                             #
01/04 |                                      #
01/09 |                                            #
01/12 |                                                  #
01/17 |                                                #
01/22 |                               #
01/25 |                         #
01/30 |                          #
02/02 |                   #
02/07 |                  #
02/12 |             #
02/15 |                 #
02/20 |                 #
02/23 |       #
02/28 |           #
03/04 |   #
03/07 |       #
03/12 |  #
03/15 |      #
03/20 |      #
------------------------------------------------------------
Min: $86.67


---
## Comparacion de las Tres Acciones

In [21]:
print("=" * 70)
print("         COMPARACION DE ACCIONES")
print("=" * 70)

acciones = [
    (PRECIOS_ACCION, "ACME Corp"),
    (ACCION_VOLATIL, "VolatilTech"),
    (ACCION_BAJISTA, "DeclineCorp")
]

for precios, nombre in acciones:
    reporte_acc = generar_reporte_completo(precios, nombre)
    rend = reporte_acc['rendimientos']
    print(f"\n{nombre}:")
    print(f"  Precio actual:    ${reporte_acc['estadisticas']['precio_actual']:.2f}")
    print(f"  Rendimiento:      {rend['rendimiento_total']:+.2f}%")
    print(f"  Volatilidad:      {reporte_acc['volatilidad']}")
    print(f"  Tendencia:        {reporte_acc['tendencia']}")
    print(f"  Senal actual:     {reporte_acc['senal_actual']}")
    print(f"  Dias positivos:   {rend['dias_positivos']}")
    print(f"  Dias negativos:   {rend['dias_negativos']}")

         COMPARACION DE ACCIONES

ACME Corp:
  Precio actual:    $92.74
  Rendimiento:      -7.75%
  Volatilidad:      MEDIA
  Tendencia:        ALCISTA
  Senal actual:     COMPRA
  Dias positivos:   26
  Dias negativos:   33

VolatilTech:
  Precio actual:    $59.42
  Rendimiento:      +33.35%
  Volatilidad:      MUY ALTA
  Tendencia:        ALCISTA
  Senal actual:     COMPRA
  Dias positivos:   30
  Dias negativos:   29

DeclineCorp:
  Precio actual:    $138.97
  Rendimiento:      -33.81%
  Volatilidad:      MEDIA
  Tendencia:        BAJISTA
  Senal actual:     MANTENER
  Dias positivos:   22
  Dias negativos:   37


---
## BONUS: RSI y Backtesting

In [22]:
def calcular_rsi(precios: pd.Series, periodos: int = 14) -> pd.Series:
    """
    Calcula el RSI (Relative Strength Index).

    Separamos los movimientos del precio en ganancias (cambios positivos)
    y perdidas (cambios negativos en valor absoluto). Calculamos el
    promedio movil de cada uno y dividimos para obtener RS. Finalmente
    aplicamos la formula del RSI que lo acota entre 0 y 100.

    RSI = 100 - (100 / (1 + RS))
    RS  = promedio_ganancias / promedio_perdidas

    Interpretacion:
    - RSI > 70: Sobrecomprado (posible caida)
    - RSI < 30: Sobrevendido (posible subida)

    Retorna: Serie con valores RSI entre 0 y 100
    """
    delta = precios.diff()

    # Separamos ganancias y perdidas; el resto queda en cero
    ganancias = delta.clip(lower=0)
    perdidas = -delta.clip(upper=0)  # Lo convertimos a positivo

    # Promedio movil de ganancias y perdidas
    avg_ganancia = ganancias.rolling(window=periodos).mean()
    avg_perdida = perdidas.rolling(window=periodos).mean()

    # Evitamos division por cero cuando no hubo perdidas
    rs = avg_ganancia / avg_perdida.replace(0, float('nan'))

    rsi = 100 - (100 / (1 + rs))
    return rsi

In [23]:
def backtest_estrategia(precios: pd.Series, senales: pd.Series, capital_inicial: float = 10000) -> Dict:
    """
    Simula la estrategia de trading y calcula el rendimiento historico.

    Recorremos dia a dia las senales. Al recibir COMPRA, invertimos
    todo el capital disponible al precio de ese dia. Al recibir VENTA,
    liquidamos la posicion. Al final del periodo cerramos cualquier
    posicion abierta al ultimo precio disponible.

    Retorna:
        {
            "capital_final": float,
            "rendimiento_total": float,
            "num_operaciones": int,
            "operaciones_ganadoras": int
        }
    """
    capital = float(capital_inicial)
    acciones_en_cartera = 0.0
    precio_compra = 0.0
    operaciones = []

    for fecha, senal in senales.items():
        precio = precios[fecha]

        if senal == "COMPRA" and acciones_en_cartera == 0 and capital > 0:
            # Compramos con todo el capital disponible
            acciones_en_cartera = capital / precio
            precio_compra = precio
            capital = 0.0

        elif senal == "VENTA" and acciones_en_cartera > 0:
            # Vendemos toda la posicion
            capital = acciones_en_cartera * precio
            ganancia = (precio - precio_compra) * acciones_en_cartera
            operaciones.append(ganancia > 0)
            acciones_en_cartera = 0.0

    # Cerramos la posicion al ultimo precio si quedamos comprados
    if acciones_en_cartera > 0:
        capital = acciones_en_cartera * precios.iloc[-1]
        ganancia = (precios.iloc[-1] - precio_compra) * acciones_en_cartera
        operaciones.append(ganancia > 0)

    rendimiento_total = ((capital - capital_inicial) / capital_inicial) * 100

    return {
        "capital_final": round(capital, 2),
        "rendimiento_total": round(rendimiento_total, 2),
        "num_operaciones": len(operaciones),
        "operaciones_ganadoras": sum(operaciones)
    }

In [24]:
# Prueba del RSI
print("-- RSI de ACME Corp (ultimos 10 valores) --")
rsi = calcular_rsi(PRECIOS_ACCION, 14)
print(rsi.tail(10).round(2))
print(f"\nRSI actual: {rsi.iloc[-1]:.2f}")
if rsi.iloc[-1] > 70:
    print("Interpretacion: Sobrecomprado")
elif rsi.iloc[-1] < 30:
    print("Interpretacion: Sobrevendido")
else:
    print("Interpretacion: Zona neutral")

-- RSI de ACME Corp (ultimos 10 valores) --
2024-03-11    31.14
2024-03-12    36.88
2024-03-13    39.90
2024-03-14    42.53
2024-03-15    44.17
2024-03-18    48.43
2024-03-19    45.07
2024-03-20    45.07
2024-03-21    54.08
2024-03-22    62.28
Freq: B, Name: ACME Corp, dtype: float64

RSI actual: 62.28
Interpretacion: Zona neutral


In [25]:
# Prueba del backtest
print("-- Backtest de estrategia MA 5/20 con ACME Corp --")
senales_bt = generar_senales_trading(PRECIOS_ACCION, ma_corta=5, ma_larga=20)
resultado_bt = backtest_estrategia(PRECIOS_ACCION, senales_bt, capital_inicial=10000)

print(f"  Capital inicial:        $10,000.00")
print(f"  Capital final:          ${resultado_bt['capital_final']:,.2f}")
print(f"  Rendimiento total:      {resultado_bt['rendimiento_total']:+.2f}%")
print(f"  Total de operaciones:   {resultado_bt['num_operaciones']}")
print(f"  Operaciones ganadoras:  {resultado_bt['operaciones_ganadoras']}")

-- Backtest de estrategia MA 5/20 con ACME Corp --
  Capital inicial:        $10,000.00
  Capital final:          $10,303.30
  Rendimiento total:      +3.03%
  Total de operaciones:   1
  Operaciones ganadoras:  1
